# Chapter 6: Audio Generation
## Transcribing an audio file with Transformers

We're first going to transcribe our first audio file. This is often the first step in any audio-processing pipeline. We'll use the Hugging Face transformers library, which provides a simple pipeline interface that downloads and runs models like Whisper for us.

First, you'll need to install the necessary libraries in your Python environment:

In [1]:
# Install the transformers library from Hugging Face
# and torch, the deep learning framework it runs on
! pip install -qU transformers torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0 requires torch==2.9.0, but you have torch 2.12.0 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Now, let's create a simple Python script to transcribe an audio file. For this example, we will use a sample audio file hosted by Hugging Face, so you can run this code immediately without needing your own file.

In [ ]:
import torch
from transformers import pipeline

# Set the device to use. 'cuda:0' uses a GPU if available (much faster),
# otherwise it falls back to 'cpu'.
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# 1. Load the ASR pipeline
# This will download the model and tokenizer for 'openai/whisper-large-v3'
# We specify the device to run the model on.
print("Loading ASR pipeline...")
transcriber = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch.float16,  # Use float16 for faster inference
    device=device,
)

# 2. Define the audio file to transcribe
# This is a dummy audio file from a Hugging Face dataset
audio_file = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
print(f"Transcribing audio from: {audio_file}")

# 3. Transcribe the audio
# The transcriber handles downloading, pre-processing, and running the model
# 'chunk_length_s=30' breaks long audio into 30-second chunks
transcription = transcriber(
    audio_file, 
    chunk_length_s=30, 
    return_timestamps=True
)

# 4. Print the result
print("\n--- Transcription Result ---")
print(transcription["text"])
print("\n--- Timestamps ---")
print(transcription["chunks"])


## Generating voice with cloud-based and local TTS models

For enterprise applications where reliability, scalability, and quality are paramount, using a managed cloud API is often the best choice. Let's see how to use the Google Cloud Text-to-Speech API to generate a high-quality voiceover. First, you'll need to install the Google Cloud library and authenticate with your Google Cloud project.

In [ ]:
# Install the Google Cloud Text-to-Speech client library
! pip install -qU google-cloud-texttospeech


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Log in with your Google Cloud user account
! gcloud auth application-default login
# And set your quota project
! gcloud auth application-default set-quota-project YOUR_PROJECT_HERE


Now, let's write a script to generate an MP3 file from a line of text. If you have not used the Cloud Text-to-Speech API before, it may ask you to activate the API. Review https://docs.cloud.google.com/text-to-speech/docs/list-voices-and-types for the list of voices that you can use with the API.

In [ ]:
from google.cloud import texttospeech

# 1. Instantiate a client
client = texttospeech.TextToSpeechClient()

# 2. Set the text input to be synthesized
text = "Hello, and welcome to our product launch. We are excited to show you what we've been working on."
synthesis_input = texttospeech.SynthesisInput(text=text)

# 3. Build the voice request
# Select the language code (e.g., "en-US") and a specific voice name.
# You can also use SSML (Speech Synthesis Markup Language) for more control (if supported by the voice)]
# Review https://docs.cloud.google.com/text-to-speech/docs/list-voices-and-types for the list of voices
voice = texttospeech.VoiceSelectionParams(
    language_code="en-US",
    name="en-US-Chirp-HD-F",  # A high-quality, studio voice
)

# 4. Select the type of audio file you want
audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.MP3
)

# 5. Perform the text-to-speech request
print(f"Generating audio for text: '{text}'")
response = client.synthesize_speech(
    input=synthesis_input, 
    voice=voice, 
    audio_config=audio_config
)

# 6. Save the audio to a file
output_file = "output_google_tts.mp3"
with open(output_file, "wb") as out:
    # The response's audio_content is binary bytes
    out.write(response.audio_content)
    
print(f"Audio content written to file '{output_file}'")

Generating audio for text: 'Hello, and welcome to our product launch. We are excited to show you what we've been working on.'
Audio content written to file 'output_voiceover.mp3'


What if you need to run TTS offline, or you want to experiment with fine-tuning your own voice models without relying on a cloud provider? Open-source libraries are an excellent choice. The `Transfomers` package already comes with a number of pre-built models. For advanced models, we need to install the phonemizer package.

Now, let's run a simple script to generate a .wav file locally. This script is very simple. We import a Transformer model, load a pre-trained model (in this case `facebook/mms-tts-eng`) and run the pipeline against our text. The library handles everything, and you now have a wav file generated entirely on your own machine. This is perfect for batch processing or privacy-sensitive applications.

In [ ]:
import torch
from transformers import pipeline, set_seed
from scipy.io.wavfile import write

# 1. Set seed for your transformer pipeline
set_seed(555)

# 2. Set the text input to be synthesized
text = "Hello, and welcome to our product launch. We are excited to show you what we've been working on.""

# 3. Instantiate the pipeline with a model
# kakao-enterprise/vits-ljs is a good quality VITS model
# alternatives like facebook/mms-tts-eng are much faster, but worse quality
pipe = pipeline(
    task="text-to-speech",
    model="facebook/mms-tts-eng",
    dtype=torch.float16,
    device=0
)

# 3. Run the pipeline!
speech = pipe(text)

# 4. Extract audio data and sampling rate
audio_data = speech["audio"]
sampling_rate = speech["sampling_rate"]

# 5. Save as WAV file
output_file = "output_local_tts.wav"
write(output_file, sampling_rate, audio_data.squeeze())
print(f"Audio content written to file '{output_file}'")

Device set to use mps:0


Audio content written to file 'output_local_tts.wav'


## Generating music and sound effects
Beyond the human voice, generative AI is now capable of composing novel, high-fidelity music and realistic sound effects from text prompts. This new capability can transform marketing, game development, and content creation. Let's create our own music! This is a fantastic way to understand the power of these models. We will once again use the Hugging Face transformers pipeline, this time with Meta's MusicGen model, to generate a short clip for a hypothetical marketing video. You will need an additional library, scipy, to save the audio file in the correct format.


In [ ]:
# Install scipy, which we need to save the.wav file
! pip install -qU scipy


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


Now, let's run the Python script.

In [ ]:
import torch
from transformers import pipeline
import scipy.io.wavfile as wavfile
import numpy as np

# 1. Set the device
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# 2. Load the text-to-audio pipeline
# This will download the 'facebook/musicgen-large' model
print("Loading MusicGen pipeline...")
synthesiser = pipeline(
    "text-to-audio", 
    "facebook/musicgen-large",
    torch_dtype=torch.float16,
    device=device
)

# 3. Define your prompt
# Be descriptive! Include genre, mood, and instrumentation.
prompt = "A 15-second, upbeat corporate jingle. Starts with a ukulele strum, then adds a light, happy drum beat and a simple piano melody."

# 4. Generate the music
# 'forward_params' lets us control generation. max_new_tokens controls length.
# 256 tokens = ~5 seconds, so 768 tokens = ~15 seconds.
print(f"Generating music for prompt: '{prompt}'")
music = synthesiser(
    prompt, 
    forward_params={"max_new_tokens": 768}
)

# 5. Save the audio to a file
output_file = "musicgen_jingle.wav"
sampling_rate = music["sampling_rate"]
audio_data = music["audio"]

# Ensure audio data is in the correct format for saving (int16)
audio_data_int16 = (np.array(audio_data) * 32767).astype(np.int16)
# If it's stereo (2 channels), transpose it
if audio_data_int16.ndim > 1:
    audio_data_int16 = audio_data_int16.T

wavfile.write(output_file, rate=sampling_rate, data=audio_data_int16)
print(f"Music written to file '{output_file}'")


Loading MusicGen pipeline...


Loading checkpoint shards: 100%|██████████| 2/2 [00:18<00:00,  9.27s/it]
Device set to use cpu
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
past_key_values should not be None in from_legacy_cache()


Generating music for prompt: 'A 15-second, upbeat corporate jingle. Starts with a ukulele strum, then adds a light, happy drum beat and a simple piano melody.'
Music written to file 'musicgen_jingle.wav'


Congrats! You've just generated a completely novel piece of music with a few lines of code. You can experiment by changing the prompt to be "lo-fi hip hop beat for studying" or "epic cinematic orchestral trailer music" and see how the model's output changes. This simple script is the engine for the new wave of on-demand, royalty-free content.